# 4주차 · 반도체 공정 불량 예측

590개 센서 값으로 그 웨이퍼가 **불량(1)인지 정상(0)인지** 예측합니다. 불량이 6.6%뿐인 **불균형** 데이터라 '전부 정상' 찍기로는 F1이 낮습니다.

**진행 방법**: 셀을 순서대로 실행하고 `✏️ 여기를 채우세요` 셀만 수정하세요.

평가 지표: **F1** (높을수록 좋음)

## 1. 데이터 불러오기  *(그냥 실행하세요)*

In [ ]:
import pandas as pd

TRAIN_URL = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/manufacturing/train.csv"  # 운영진이 채워둡니다
TEST_URL  = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/manufacturing/test.csv"

if TRAIN_URL:
    train = pd.read_csv(TRAIN_URL); test = pd.read_csv(TEST_URL)
else:
    from google.colab import files; files.upload()
    train = pd.read_csv("train.csv"); test = pd.read_csv("test.csv")

sensor_cols = [c for c in train.columns if c not in ("id", "target")]
print("train:", train.shape, "| test:", test.shape)
print("불량 비율:", round(train["target"].mean(), 3))

## 2. ✏️ 여기를 채우세요 — 전처리 & 모델
동작하는 기본 예시(결측치 중앙값 대치 + 스케일링 + 로지스틱 회귀)입니다. `class_weight='balanced'`로 불균형에 대응해 '전부 정상' 찍기(F1 0.48)를 넘습니다.

점수를 올리려면: 랜덤포레스트 등 다른 모델, 피처 선택, **예측 임계값(threshold) 낮추기**(불량을 더 적극적으로 잡기), 스케일/결측 전처리 바꾸기.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 결측치 대치 → 스케일링 → 로지스틱 회귀 (불균형 대응)
# ✏️ 모델이나 전처리를 바꿔가며 실험해보세요
clf = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced"),
)
clf.fit(train[sensor_cols], train["target"])

## 3. 예측 & 제출 저장  *(실행하면 자동 다운로드)*

In [ ]:
test["prediction"] = clf.predict(test[sensor_cols])
test[["id", "prediction"]].to_csv("submission.csv", index=False)
print("저장 완료 · 예측된 불량 수:", int(test["prediction"].sum()))
try:
    from google.colab import files; files.download("submission.csv")
except Exception:
    print("왼쪽 파일탭에서 submission.csv를 내려받으세요.")